<a href="https://colab.research.google.com/github/PowerRanger18/food-image-recognition/blob/main/food_image_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install torch torchvision

In [8]:
import torch
import torchvision
import torchvision.models as models
import torchvision.transforms as transforms

from PIL import Image
import requests
from io import BytesIO
print("step 2 works")

step 2 works


ResNet-18
~11 million parameters
ResNet-50
~25 million parameters

In [9]:
model = models.resnet50(pretrained=True)
model.eval()

print("Model loaded successfully")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 101MB/s] 


Model loaded successfully


In [10]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Transform ready")

Transform ready


In [11]:
from google.colab import files

def load_image_from_upload():
    uploaded = files.upload()
    image = Image.open(list(uploaded.keys())[0]).convert("RGB")
    return image

In [12]:
def predict(image):
    input_tensor = transform(image).unsqueeze(0)

    with torch.no_grad():
        output = model(input_tensor)

    prediction = output.argmax(dim=1).item()
    return prediction

In [13]:
image = load_image_from_upload()

result = predict(image)
print("Prediction class ID:", result)

Saving Rice_grains_(IRRI).jpg to Rice_grains_(IRRI) (2).jpg
Prediction class ID: 509


Load Food-101 dataset

In [14]:
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

train_data = datasets.Food101(root='./data', split='train', download=True, transform=transform)
test_data = datasets.Food101(root='./data', split='test', download=True, transform=transform)

100%|██████████| 5.00G/5.00G [04:09<00:00, 20.1MB/s]


Train (fine-tune)
A pretrained ResNet (trained on ImageNet dataset) has already learned:

Early layers → edges, textures, colors
Middle layers → shapes, patterns
Deep layers → object concepts

Modify ResNet output layer

In [15]:
import torchvision.models as models
import torch.nn as nn

model = models.resnet50(pretrained=True)

# Replace final layer
model.fc = nn.Linear(model.fc.in_features, 101)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [16]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [18]:
from torch.utils.data import DataLoader

test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

In [20]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, preds = outputs.max(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

print("Test Accuracy:", correct / total)

KeyboardInterrupt: 